# AMR-EnsembleNet — CNN/XGBoost Weight Sweep

The authors' own ensemble notebook (`AMR Ensemble Models/AMR_Project_Ensemble_Soft_Voting.ipynb`)
only ever evaluates a single, hand-picked blend: `cnn_weight = xgb_weight = 0.5`. The paper reports
this 50/50 split without showing that it is actually the best-performing blend — it's asserted, not
searched for.

This notebook sweeps the blend weight from a pure-XGBoost ensemble (`cnn_weight = 0.0`) to a
pure-CNN ensemble (`cnn_weight = 1.0`) in steps of 0.05, for **all four antibiotics** (CIP, CTX, CTZ,
GEN), using the exact same saved models and identical 80/20 stratified split (`random_state=42`) as
the authors' notebook — so results are directly comparable to their reported 50/50 numbers.

For each weight point we report:
* **AUC** — threshold-independent, the cleanest signal for "is this blend better," since it doesn't
  depend on any threshold choice.
* **Accuracy, MCC, Macro F1, Recall (resistant class)** — computed at each weight's own
  Youden's-J-optimal threshold (`argmax(tpr - fpr)` off that weight's ROC curve), analogous to how
  the original notebook hand-picked a per-antibiotic threshold at the 50/50 point. This isolates the
  effect of the *blend weight* from the effect of *threshold choice*.

**Prerequisite (resolved 2026-09-27):** `best_xgboost_model_{ANTIBIOTIC}.json` for all four
antibiotics, plus the full `Giessen_dataset/` CSVs, are now committed directly in the repo under
`results/` and `Giessen_dataset/` — no Google Drive mount needed. This notebook works straight off
a clone of `oliGAMER/ML_Paper_Reprod`.


## 1. Get the repo

If this is already running from inside a checkout of the repo (e.g. you opened this file locally, or
via Colab's GitHub integration against the repo itself), this cell is a no-op. Otherwise it clones
the repo fresh — no Drive mount, no manual uploads.

In [ ]:
import os

REPO_URL = "https://github.com/oliGAMER/ML_Paper_Reprod.git"
REPO_DIR = "/content/ML_Paper_Reprod"

if os.path.isdir("Giessen_dataset") and os.path.isdir("results"):
    print(f"Already inside the repo checkout: {os.getcwd()}")
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"Repo already cloned, switched to {REPO_DIR}")
else:
    get_ipython().system(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)
    print(f"Cloned repo, switched to {REPO_DIR}")

print("Working dir:", os.getcwd())


## 2. Setup, imports, config

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score, roc_curve,
    matthews_corrcoef, recall_score
)
import tensorflow as tf
from tensorflow import keras
import xgboost as xgb

np.random.seed(42)
tf.random.set_seed(42)

SNP_MATRIX_FILE = "Giessen_dataset/cip_ctx_ctz_gen_multi_data.csv"
PHENOTYPES_FILE = "Giessen_dataset/cip_ctx_ctz_gen_pheno.csv"
WEIGHTS_DIR = "results"

ANTIBIOTICS = ["CIP", "CTX", "CTZ", "GEN"]
TEST_SPLIT = 0.2

# The blend weight to sweep: final_prob = w * cnn_prob + (1 - w) * xgb_prob
SWEEP_WEIGHTS = np.round(np.arange(0.0, 1.0001, 0.05), 2)

# Paper / authors' notebook reference point, for comparison
PAPER_CNN_WEIGHT = 0.5

print(f"Sweeping {len(SWEEP_WEIGHTS)} weight points from 0.0 (pure XGBoost) to 1.0 (pure CNN)")
print(f"for antibiotics: {ANTIBIOTICS}")


## 3. Helpers

Identical data-loading/splitting logic to the authors' ensemble notebook, so the test set here
matches theirs exactly.

In [ ]:
def load_split(antibiotic):
    snp_df_raw = pd.read_csv(SNP_MATRIX_FILE)
    pheno_df = pd.read_csv(PHENOTYPES_FILE)
    snp_df = snp_df_raw.set_index('prename')
    pheno_df = pheno_df.set_index('prename')
    merged_df = snp_df.join(pheno_df[antibiotic], how='inner')
    merged_df = merged_df.dropna(subset=[antibiotic])
    merged_df = merged_df.fillna(0)

    labels = merged_df[antibiotic].values.astype(np.int32)
    X = merged_df.drop(columns=[antibiotic]).values.astype(np.int32)

    X_train, X_test, y_train, y_test = train_test_split(
        X, labels, test_size=TEST_SPLIT, random_state=42, stratify=labels
    )
    return X_test, y_test


def load_models(antibiotic):
    cnn_path = os.path.join(WEIGHTS_DIR, f"best_cnn1d_model_{antibiotic}.keras")
    xgb_path = os.path.join(WEIGHTS_DIR, f"best_xgboost_model_{antibiotic}.json")

    cnn_model = keras.models.load_model(cnn_path)
    xgb_model = xgb.XGBClassifier()
    xgb_model.load_model(xgb_path)
    return cnn_model, xgb_model


def youden_threshold(y_true, probs):
    """Threshold maximizing tpr - fpr on this weight's own ROC curve."""
    fpr, tpr, thresholds = roc_curve(y_true, probs)
    j = tpr - fpr
    return thresholds[np.argmax(j)]


def metrics_at_weight(y_test, probs_cnn, probs_xgb, w):
    probs = w * probs_cnn + (1.0 - w) * probs_xgb
    thr = youden_threshold(y_test, probs)
    preds = (probs >= thr).astype(int)

    return {
        "cnn_weight": w,
        "xgb_weight": round(1.0 - w, 2),
        "threshold": thr,
        "accuracy": accuracy_score(y_test, preds),
        "mcc": matthews_corrcoef(y_test, preds),
        "macro_f1": f1_score(y_test, preds, average="macro"),
        "recall_resistant": recall_score(y_test, preds, pos_label=1),
        "auc": roc_auc_score(y_test, probs),  # threshold-independent
    }


## 4. Run the sweep for all four antibiotics

In [ ]:
sweep_results = {}

for antibiotic in ANTIBIOTICS:
    print(f"\n--- Sweeping {antibiotic} ---")
    X_test, y_test = load_split(antibiotic)
    cnn_model, xgb_model = load_models(antibiotic)

    probs_cnn = cnn_model.predict(X_test, verbose=0).ravel()
    probs_xgb = xgb_model.predict_proba(X_test)[:, 1]

    rows = [metrics_at_weight(y_test, probs_cnn, probs_xgb, w) for w in SWEEP_WEIGHTS]
    df = pd.DataFrame(rows)
    sweep_results[antibiotic] = df

    best_mcc_row = df.loc[df["mcc"].idxmax()]
    best_auc_row = df.loc[df["auc"].idxmax()]
    print(f"  Best MCC: {best_mcc_row['mcc']:.4f} at cnn_weight={best_mcc_row['cnn_weight']:.2f} "
          f"(paper uses cnn_weight={PAPER_CNN_WEIGHT})")
    print(f"  Best AUC: {best_auc_row['auc']:.4f} at cnn_weight={best_auc_row['cnn_weight']:.2f}")

    os.makedirs("results", exist_ok=True)
    df.to_csv(f"results/{antibiotic}_weight_sweep.csv", index=False)


## 5. Plot each metric vs. blend weight, per antibiotic

In [ ]:
METRICS_TO_PLOT = ["accuracy", "mcc", "macro_f1", "recall_resistant", "auc"]

os.makedirs("results/plots", exist_ok=True)

for antibiotic, df in sweep_results.items():
    fig, ax = plt.subplots(figsize=(9, 6))
    for metric in METRICS_TO_PLOT:
        ax.plot(df["cnn_weight"], df[metric], marker="o", label=metric)

    # Mark the paper's fixed 50/50 point
    ax.axvline(PAPER_CNN_WEIGHT, color="gray", linestyle="--", alpha=0.7,
               label=f"paper (cnn_weight={PAPER_CNN_WEIGHT})")

    # Mark this antibiotic's own MCC-optimal weight
    best_row = df.loc[df["mcc"].idxmax()]
    ax.axvline(best_row["cnn_weight"], color="red", linestyle=":", alpha=0.7,
               label=f"MCC-optimal (cnn_weight={best_row['cnn_weight']:.2f})")

    ax.set_xlabel("CNN weight  (xgb_weight = 1 - cnn_weight)")
    ax.set_ylabel("Metric value")
    ax.set_title(f"{antibiotic}: Ensemble metrics vs. blend weight")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"results/plots/{antibiotic}_weight_sweep.png", dpi=150)
    plt.show()


## 6. Summary: is 50/50 actually optimal?

Per-antibiotic optimal weight (by MCC and by AUC) vs. the paper's fixed 50/50, plus the metric gap
this would buy if the ensemble weight were tuned instead of fixed.

In [ ]:
summary_rows = []
for antibiotic, df in sweep_results.items():
    paper_row = df.iloc[(df["cnn_weight"] - PAPER_CNN_WEIGHT).abs().idxmin()]
    best_mcc_row = df.loc[df["mcc"].idxmax()]
    best_auc_row = df.loc[df["auc"].idxmax()]

    summary_rows.append({
        "antibiotic": antibiotic,
        "paper_cnn_weight": PAPER_CNN_WEIGHT,
        "paper_mcc": paper_row["mcc"],
        "best_mcc_weight": best_mcc_row["cnn_weight"],
        "best_mcc": best_mcc_row["mcc"],
        "mcc_gain": best_mcc_row["mcc"] - paper_row["mcc"],
        "best_auc_weight": best_auc_row["cnn_weight"],
        "best_auc": best_auc_row["auc"],
        "auc_gain": best_auc_row["auc"] - paper_row["auc"],
    })

summary_df = pd.DataFrame(summary_rows).set_index("antibiotic")
summary_df.to_csv("results/weight_sweep_summary.csv")
summary_df


## 7. Save results back to the repo

The CSVs and PNGs above were already written into `results/` and `results/plots/` in this checkout.
To share them with the team: `File → Save a copy in GitHub` (pick a branch, not straight to `main`,
and write a real commit message — e.g. "Add CNN/XGBoost weight-sweep results and plots"), or if
you're running this locally, just `git add`/`git commit`/`git push` the new files under `results/`
the normal way.

In [ ]:
import os

print("New/updated files ready to commit:")
for f in sorted(os.listdir("results")):
    if "weight_sweep" in f:
        print(" -", os.path.join("results", f))
for f in sorted(os.listdir("results/plots")):
    if "weight_sweep" in f:
        print(" -", os.path.join("results/plots", f))
